In [1]:
pip install flask flask-cors requests -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sqlite3, os, sys, json, threading, time, webbrowser, socket
from datetime import datetime
from flask import Flask, request, jsonify
from flask_cors import CORS

# ---------- FIXED ABSOLUTE PATH — same every session ----------
DB_DIR = os.path.expanduser('~/sales_tracker')
os.makedirs(DB_DIR, exist_ok=True)
DB_FILE = os.path.join(DB_DIR, 'sales_invoice.db')

print(f"📁 DB file: {DB_FILE}")
if os.path.exists(DB_FILE):
    print(f"✅ Existing DB found ({os.path.getsize(DB_FILE)/1024:.1f} KB) — data preserved")
else:
    print("🆕 No DB yet — will create fresh")

# ---------- NEVER DELETE THE DB HERE ----------

def get_db():
    c = sqlite3.connect(DB_FILE, timeout=10)
    c.row_factory = sqlite3.Row
    c.execute('PRAGMA foreign_keys = ON')
    return c

conn = get_db()
conn.executescript('''
CREATE TABLE IF NOT EXISTS customers (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  name TEXT NOT NULL, email TEXT DEFAULT '', phone TEXT DEFAULT '',
  address TEXT DEFAULT '', created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);

CREATE TABLE IF NOT EXISTS products (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  name TEXT NOT NULL, description TEXT DEFAULT '',
  price REAL NOT NULL DEFAULT 0, stock INTEGER DEFAULT 0,
  category TEXT DEFAULT 'General',
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);

CREATE TABLE IF NOT EXISTS invoices (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  invoice_number TEXT UNIQUE NOT NULL, customer_id INTEGER NOT NULL,
  invoice_date TEXT NOT NULL, due_date TEXT DEFAULT '',
  subtotal REAL DEFAULT 0, tax REAL DEFAULT 0, discount REAL DEFAULT 0,
  total REAL DEFAULT 0, status TEXT DEFAULT 'Pending', notes TEXT DEFAULT '',
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
  FOREIGN KEY (customer_id) REFERENCES customers(id));

CREATE TABLE IF NOT EXISTS invoice_items (
  id INTEGER PRIMARY KEY AUTOINCREMENT, invoice_id INTEGER NOT NULL,
  product_id INTEGER, product_name TEXT NOT NULL,
  quantity INTEGER NOT NULL, unit_price REAL NOT NULL, total REAL NOT NULL,
  FOREIGN KEY (invoice_id) REFERENCES invoices(id));

CREATE TABLE IF NOT EXISTS orders (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  order_number TEXT UNIQUE NOT NULL,
  customer_name TEXT NOT NULL, customer_email TEXT DEFAULT '',
  customer_phone TEXT DEFAULT '', customer_address TEXT DEFAULT '',
  subtotal REAL DEFAULT 0, tax REAL DEFAULT 0, discount REAL DEFAULT 0,
  total REAL DEFAULT 0, status TEXT DEFAULT 'Pending',
  invoice_id INTEGER, notes TEXT DEFAULT '',
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);

CREATE TABLE IF NOT EXISTS order_items (
  id INTEGER PRIMARY KEY AUTOINCREMENT, order_id INTEGER NOT NULL,
  product_id INTEGER, product_name TEXT NOT NULL,
  quantity INTEGER NOT NULL, unit_price REAL NOT NULL, total REAL NOT NULL,
  FOREIGN KEY (order_id) REFERENCES orders(id));
''')

# Seed products only if table is empty
n = conn.execute('SELECT COUNT(*) FROM products').fetchone()[0]
if n == 0:
    seed = [
        ('Wireless Mouse',      'Ergonomic 2.4GHz wireless mouse',        19.99, 100, 'Accessories'),
        ('Mechanical Keyboard', 'RGB backlit mechanical keyboard',        89.99,  50, 'Accessories'),
        ('USB-C Hub 7-in-1',    'HDMI + USB3 + SD reader',                39.99,  75, 'Accessories'),
        ('27" LED Monitor',     'FHD IPS 75Hz 1920x1080',                199.99,  30, 'Displays'),
        ('Laptop Stand',        'Aluminum adjustable stand',              29.99,  60, 'Accessories'),
        ('Noise-Cancel Headset','Over-ear Bluetooth 5.0 30h battery',    129.99,  40, 'Audio'),
        ('Webcam 1080p',        'Auto-focus + dual mic',                  59.99,  55, 'Video'),
        ('Portable SSD 1TB',    'USB-C 1050MB/s',                        109.99,  45, 'Storage'),
        ('Desk Lamp LED',       'Touch dimmable 3 color temp',            24.99,  80, 'Home Office'),
        ('Wireless Charger',    '15W Qi fast-charge pad',                 22.99,  90, 'Accessories'),
    ]
    conn.executemany(
        'INSERT INTO products (name, description, price, stock, category) VALUES (?,?,?,?,?)', seed)
    conn.commit()
    print("✅ Seeded 10 products")
else:
    print(f"✅ Existing DB: {n} products")

counts = {
    'customers': conn.execute('SELECT COUNT(*) FROM customers').fetchone()[0],
    'products':  conn.execute('SELECT COUNT(*) FROM products').fetchone()[0],
    'invoices':  conn.execute('SELECT COUNT(*) FROM invoices').fetchone()[0],
    'orders':    conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0],
}
print("📊 DB contents:", counts)

last = conn.execute('SELECT invoice_number, total, status FROM invoices ORDER BY id DESC LIMIT 5').fetchall()
if last:
    print("🧾 Last invoices in DB:")
    for r in last:
        print(f"   • {r['invoice_number']}  ${r['total']:.2f}  [{r['status']}]")
else:
    print("🧾 No invoices stored yet")

conn.close()

📁 DB file: C:\Users\AZIZUR RAHAMAN/sales_tracker\sales_invoice.db
✅ Existing DB found (40.0 KB) — data preserved
✅ Existing DB: 10 products
📊 DB contents: {'customers': 4, 'products': 10, 'invoices': 4, 'orders': 4}
🧾 Last invoices in DB:
   • INV-202611-0004  $262.95  [Paid]
   • INV-202611-0003  $49.98  [Paid]
   • INV-202610-0002  $49.98  [Paid]
   • INV-202610-0001  $59.98  [Paid]


In [3]:
def get_local_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(('8.8.8.8', 80))
        ip = s.getsockname()[0]
    except Exception:
        ip = '127.0.0.1'
    finally:
        s.close()
    return ip

app = Flask(__name__)
CORS(app, resources={r"/api/*": {"origins": "*"}})
app.config['JSON_SORT_KEYS'] = False

@app.errorhandler(Exception)
def _handle(e):
    import traceback
    traceback.print_exc()
    return jsonify({'error': str(e), 'type': type(e).__name__}), 500

LOCAL_IP = get_local_ip()
print(f"✅ Flask app created")
print(f"💻 LAN IP: http://{LOCAL_IP}:5000")

✅ Flask app created
💻 LAN IP: http://192.168.29.57:5000


In [4]:
# ---------- CUSTOMERS ----------
@app.route('/api/customers', methods=['GET'])
def customers_list():
    conn = get_db()
    rows = conn.execute('SELECT * FROM customers ORDER BY name').fetchall()
    conn.close()
    return jsonify([dict(r) for r in rows])

@app.route('/api/customers', methods=['POST'])
def customers_create():
    d = request.get_json(silent=True) or {}
    name = (d.get('name') or '').strip()
    if not name:
        return jsonify({'error': 'name is required'}), 400
    conn = get_db()
    try:
        cur = conn.execute(
            'INSERT INTO customers (name, email, phone, address) VALUES (?,?,?,?)',
            (name, d.get('email',''), d.get('phone',''), d.get('address','')))
        conn.commit()
        return jsonify({'id': cur.lastrowid, 'name': name}), 201
    finally:
        conn.close()

@app.route('/api/customers/<int:cid>', methods=['PUT'])
def customers_update(cid):
    d = request.get_json(silent=True) or {}
    name = (d.get('name') or '').strip()
    if not name:
        return jsonify({'error': 'name is required'}), 400
    conn = get_db()
    try:
        conn.execute('UPDATE customers SET name=?, email=?, phone=?, address=? WHERE id=?',
                     (name, d.get('email',''), d.get('phone',''), d.get('address',''), cid))
        conn.commit()
        return jsonify({'ok': True})
    finally:
        conn.close()

@app.route('/api/customers/<int:cid>', methods=['DELETE'])
def customers_delete(cid):
    conn = get_db()
    try:
        conn.execute('DELETE FROM customers WHERE id=?', (cid,))
        conn.commit()
        return jsonify({'ok': True})
    finally:
        conn.close()

# ---------- PRODUCTS ----------
@app.route('/api/products', methods=['GET'])
def products_list():
    conn = get_db()
    rows = conn.execute('SELECT * FROM products ORDER BY name').fetchall()
    conn.close()
    return jsonify([dict(r) for r in rows])

@app.route('/api/products', methods=['POST'])
def products_create():
    d = request.get_json(silent=True) or {}
    name = (d.get('name') or '').strip()
    if not name:
        return jsonify({'error': 'name is required'}), 400
    try:
        price = float(d.get('price', 0)); stock = int(d.get('stock', 0))
    except (TypeError, ValueError):
        return jsonify({'error': 'invalid price/stock'}), 400
    conn = get_db()
    try:
        cur = conn.execute(
            'INSERT INTO products (name, description, price, stock, category) VALUES (?,?,?,?,?)',
            (name, d.get('description',''), price, stock, d.get('category','General')))
        conn.commit()
        return jsonify({'id': cur.lastrowid, 'name': name}), 201
    finally:
        conn.close()

@app.route('/api/products/<int:pid>', methods=['PUT'])
def products_update(pid):
    d = request.get_json(silent=True) or {}
    name = (d.get('name') or '').strip()
    if not name:
        return jsonify({'error': 'name is required'}), 400
    try:
        price = float(d.get('price', 0)); stock = int(d.get('stock', 0))
    except (TypeError, ValueError):
        return jsonify({'error': 'invalid price/stock'}), 400
    conn = get_db()
    try:
        conn.execute(
            'UPDATE products SET name=?, description=?, price=?, stock=?, category=? WHERE id=?',
            (name, d.get('description',''), price, stock, d.get('category','General'), pid))
        conn.commit()
        return jsonify({'ok': True})
    finally:
        conn.close()

@app.route('/api/products/<int:pid>', methods=['DELETE'])
def products_delete(pid):
    conn = get_db()
    try:
        conn.execute('DELETE FROM products WHERE id=?', (pid,))
        conn.commit()
        return jsonify({'ok': True})
    finally:
        conn.close()

print("✅ Customer + Product routes ready")

✅ Customer + Product routes ready


In [5]:
def _next_invoice_number():
    conn = get_db()
    try:
        n = conn.execute('SELECT COUNT(*) FROM invoices').fetchone()[0] + 1
    finally:
        conn.close()
    return f"INV-{datetime.now().strftime('%Y%m')}-{n:04d}"

def _next_order_number():
    conn = get_db()
    try:
        n = conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0] + 1
    finally:
        conn.close()
    return f"ORD-{datetime.now().strftime('%Y%m')}-{n:04d}"

@app.route('/api/invoices', methods=['GET'])
def invoices_list():
    conn = get_db()
    rows = conn.execute('''SELECT i.*, c.name AS customer_name
                           FROM invoices i LEFT JOIN customers c ON i.customer_id = c.id
                           ORDER BY i.id DESC''').fetchall()
    conn.close()
    return jsonify([dict(r) for r in rows])

@app.route('/api/invoices/<int:iid>', methods=['GET'])
def invoices_get(iid):
    conn = get_db()
    inv = conn.execute('''SELECT i.*, c.name AS customer_name, c.email AS customer_email,
                          c.phone AS customer_phone, c.address AS customer_address
                          FROM invoices i LEFT JOIN customers c ON i.customer_id = c.id
                          WHERE i.id=?''', (iid,)).fetchone()
    if not inv:
        conn.close(); return jsonify({'error': 'not found'}), 404
    items = conn.execute('SELECT * FROM invoice_items WHERE invoice_id=?', (iid,)).fetchall()
    conn.close()
    out = dict(inv); out['items'] = [dict(x) for x in items]
    return jsonify(out)

@app.route('/api/invoices', methods=['POST'])
def invoices_create():
    d = request.get_json(silent=True) or {}
    if not d.get('customer_id'):
        return jsonify({'error': 'customer_id required'}), 400
    items = d.get('items') or []
    if not items:
        return jsonify({'error': 'at least one item required'}), 400
    sub = sum(float(it['quantity']) * float(it['unit_price']) for it in items)
    disc = float(d.get('discount', 0)); rate = float(d.get('tax_rate', 0))
    taxable = max(0.0, sub - disc); tax = taxable * rate / 100
    total = taxable + tax; num = _next_invoice_number()
    conn = get_db()
    try:
        cur = conn.execute('''INSERT INTO invoices
            (invoice_number, customer_id, invoice_date, due_date,
             subtotal, tax, discount, total, status, notes)
            VALUES (?,?,?,?,?,?,?,?,?,?)''',
            (num, int(d['customer_id']),
             d.get('invoice_date') or datetime.now().strftime('%Y-%m-%d'),
             d.get('due_date',''), sub, tax, disc, total,
             d.get('status','Pending'), d.get('notes','')))
        iid = cur.lastrowid
        for it in items:
            t = float(it['quantity']) * float(it['unit_price'])
            conn.execute('''INSERT INTO invoice_items
                (invoice_id, product_id, product_name, quantity, unit_price, total)
                VALUES (?,?,?,?,?,?)''',
                (iid, it.get('product_id'), it['product_name'],
                 int(it['quantity']), float(it['unit_price']), t))
            if it.get('product_id'):
                conn.execute('UPDATE products SET stock = stock - ? WHERE id=?',
                             (int(it['quantity']), it['product_id']))
        conn.commit()
        return jsonify({'id': iid, 'invoice_number': num}), 201
    except Exception:
        conn.rollback(); raise
    finally:
        conn.close()

@app.route('/api/invoices/<int:iid>/status', methods=['PUT'])
def invoices_status(iid):
    d = request.get_json(silent=True) or {}
    conn = get_db()
    try:
        conn.execute('UPDATE invoices SET status=? WHERE id=?', (d.get('status','Pending'), iid))
        conn.commit(); return jsonify({'ok': True})
    finally:
        conn.close()

@app.route('/api/invoices/<int:iid>', methods=['DELETE'])
def invoices_delete(iid):
    conn = get_db()
    try:
        conn.execute('DELETE FROM invoice_items WHERE invoice_id=?', (iid,))
        conn.execute('DELETE FROM invoices WHERE id=?', (iid,))
        conn.commit(); return jsonify({'ok': True})
    finally:
        conn.close()

@app.route('/api/dashboard', methods=['GET'])
def dashboard():
    conn = get_db()
    try:
        today = datetime.now().strftime('%Y-%m-%d')
        this_month = datetime.now().strftime('%Y-%m')
        this_year = datetime.now().strftime('%Y')
        def g(q, args=()):
            return conn.execute(q, args).fetchone()[0]
        data = {
            'revenue':   g("SELECT COALESCE(SUM(total),0) FROM invoices WHERE status='Paid'"),
            'invoices':  g("SELECT COUNT(*) FROM invoices"),
            'pending':   g("SELECT COUNT(*) FROM invoices WHERE status='Pending'"),
            'customers': g("SELECT COUNT(*) FROM customers"),
            'products':  g("SELECT COUNT(*) FROM products"),
            'orders_pending': g("SELECT COUNT(*) FROM orders WHERE status='Pending'"),
            'today_revenue':  g("SELECT COALESCE(SUM(total),0) FROM invoices WHERE invoice_date=?", (today,)),
            'month_revenue':  g("SELECT COALESCE(SUM(total),0) FROM invoices WHERE substr(invoice_date,1,7)=?", (this_month,)),
            'year_revenue':   g("SELECT COALESCE(SUM(total),0) FROM invoices WHERE substr(invoice_date,1,4)=?", (this_year,)),
            'today_invoices': g("SELECT COUNT(*) FROM invoices WHERE invoice_date=?", (today,)),
            'month_invoices': g("SELECT COUNT(*) FROM invoices WHERE substr(invoice_date,1,7)=?", (this_month,)),
            'year_invoices':  g("SELECT COUNT(*) FROM invoices WHERE substr(invoice_date,1,4)=?", (this_year,)),
        }
        data['monthly'] = [dict(r) for r in conn.execute(
            "SELECT substr(invoice_date,1,7) m, SUM(total) t FROM invoices GROUP BY m ORDER BY m DESC LIMIT 6").fetchall()]
        data['top'] = [dict(r) for r in conn.execute(
            "SELECT product_name n, SUM(quantity) q, SUM(total) r FROM invoice_items GROUP BY product_name ORDER BY r DESC LIMIT 5").fetchall()]
    finally:
        conn.close()
    return jsonify(data)

@app.route('/api/reports', methods=['GET'])
def reports():
    granularity = (request.args.get('granularity') or 'day').lower()
    start = request.args.get('start') or ''
    end   = request.args.get('end') or ''
    try:
        limit = max(1, min(365, int(request.args.get('limit', 30))))
    except ValueError:
        limit = 30
    if granularity not in ('day', 'month', 'year'):
        return jsonify({'error': 'granularity must be day|month|year'}), 400

    key = {'day': "invoice_date", 'month': "substr(invoice_date,1,7)", 'year': "substr(invoice_date,1,4)"}[granularity]
    where, args = [], []
    if start:
        where.append("invoice_date >= ?"); args.append(start)
    if end:
        where.append("invoice_date <= ?"); args.append(end)
    where_sql = ('WHERE ' + ' AND '.join(where)) if where else ''

    conn = get_db()
    try:
        inv_rows = conn.execute(f'''
            SELECT {key} AS period, SUM(total) AS revenue, COUNT(*) AS invoices,
                   SUM(CASE WHEN status='Paid' THEN total ELSE 0 END) AS paid,
                   SUM(CASE WHEN status='Pending' THEN total ELSE 0 END) AS pending
            FROM invoices {where_sql}
            GROUP BY period ORDER BY period DESC LIMIT ?
        ''', (*args, limit)).fetchall()

        order_key = {'day': "substr(created_at,1,10)",
                     'month': "substr(created_at,1,7)",
                     'year': "substr(created_at,1,4)"}[granularity]
        ord_rows = conn.execute(f'''
            SELECT {order_key} AS period, COUNT(*) AS total,
                   SUM(CASE WHEN status='Accepted' THEN 1 ELSE 0 END) AS accepted,
                   SUM(CASE WHEN status='Pending'  THEN 1 ELSE 0 END) AS pending,
                   SUM(CASE WHEN status='Rejected' THEN 1 ELSE 0 END) AS rejected
            FROM orders GROUP BY period ORDER BY period DESC LIMIT ?
        ''', (limit,)).fetchall()
        ord_map = {r['period']: dict(r) for r in ord_rows}

        rows = []
        for r in inv_rows:
            row = dict(r); o = ord_map.get(row['period'], {})
            row['orders_total']    = o.get('total', 0)
            row['orders_accepted'] = o.get('accepted', 0)
            row['orders_pending']  = o.get('pending', 0)
            row['orders_rejected'] = o.get('rejected', 0)
            rows.append(row)
        inv_periods = {r['period'] for r in inv_rows}
        for period, o in ord_map.items():
            if period not in inv_periods:
                rows.append({'period': period, 'revenue': 0, 'invoices': 0,
                             'paid': 0, 'pending': 0,
                             'orders_total': o['total'], 'orders_accepted': o['accepted'],
                             'orders_pending': o['pending'], 'orders_rejected': o['rejected']})
        rows.sort(key=lambda x: x['period'], reverse=True)
        rows = rows[:limit]
        totals = {k: sum(r[k] for r in rows) for k in
                  ('revenue','invoices','paid','pending',
                   'orders_total','orders_accepted','orders_pending','orders_rejected')}
    finally:
        conn.close()
    return jsonify({'granularity': granularity, 'start': start, 'end': end,
                    'limit': limit, 'rows': rows, 'totals': totals})

print("✅ Invoice + Dashboard + Reports routes ready")

✅ Invoice + Dashboard + Reports routes ready


In [6]:
@app.route('/api/catalog', methods=['GET'])
def catalog():
    conn = get_db()
    rows = conn.execute('''SELECT id, name, description, price, stock, category
                           FROM products WHERE stock > 0 ORDER BY category, name''').fetchall()
    conn.close()
    return jsonify([dict(r) for r in rows])

@app.route('/api/orders', methods=['POST'])
def orders_create():
    d = request.get_json(silent=True) or {}
    name = (d.get('customer_name') or '').strip()
    if not name:
        return jsonify({'error': 'customer_name is required'}), 400
    items = d.get('items') or []
    if not items:
        return jsonify({'error': 'order must contain at least one item'}), 400
    conn = get_db()
    try:
        for it in items:
            p = conn.execute('SELECT stock, name FROM products WHERE id=?',
                             (it.get('product_id'),)).fetchone()
            if not p:
                return jsonify({'error': f"Product {it.get('product_id')} not found"}), 400
            if p['stock'] < int(it['quantity']):
                return jsonify({'error': f"Not enough stock for {p['name']} (have {p['stock']})"}), 400
        sub = sum(float(it['quantity']) * float(it['unit_price']) for it in items)
        rate = float(d.get('tax_rate', 0))
        tax = sub * rate / 100
        total = sub + tax
        num = _next_order_number()
        cur = conn.execute('''INSERT INTO orders
            (order_number, customer_name, customer_email, customer_phone,
             customer_address, subtotal, tax, discount, total, status, notes)
            VALUES (?,?,?,?,?,?,?,?,?,?,?)''',
            (num, name, d.get('customer_email',''), d.get('customer_phone',''),
             d.get('customer_address',''), sub, tax, 0, total, 'Pending', d.get('notes','')))
        oid = cur.lastrowid
        for it in items:
            t = float(it['quantity']) * float(it['unit_price'])
            conn.execute('''INSERT INTO order_items
                (order_id, product_id, product_name, quantity, unit_price, total)
                VALUES (?,?,?,?,?,?)''',
                (oid, it.get('product_id'), it['product_name'],
                 int(it['quantity']), float(it['unit_price']), t))
        conn.commit()
        return jsonify({'id': oid, 'order_number': num, 'total': total}), 201
    except Exception:
        conn.rollback(); raise
    finally:
        conn.close()

@app.route('/api/orders', methods=['GET'])
def orders_list():
    status = request.args.get('status')
    conn = get_db()
    if status:
        rows = conn.execute('SELECT * FROM orders WHERE status=? ORDER BY id DESC', (status,)).fetchall()
    else:
        rows = conn.execute('SELECT * FROM orders ORDER BY id DESC').fetchall()
    conn.close()
    return jsonify([dict(r) for r in rows])

@app.route('/api/orders/<int:oid>', methods=['GET'])
def orders_get(oid):
    conn = get_db()
    o = conn.execute('SELECT * FROM orders WHERE id=?', (oid,)).fetchone()
    if not o:
        conn.close(); return jsonify({'error': 'not found'}), 404
    items = conn.execute('SELECT * FROM order_items WHERE order_id=?', (oid,)).fetchall()
    conn.close()
    out = dict(o); out['items'] = [dict(x) for x in items]
    return jsonify(out)

@app.route('/api/orders/<int:oid>/accept', methods=['POST'])
def orders_accept(oid):
    conn = get_db()
    try:
        o = conn.execute('SELECT * FROM orders WHERE id=?', (oid,)).fetchone()
        if not o:
            return jsonify({'error': 'order not found'}), 404
        if o['status'] != 'Pending':
            return jsonify({'error': f"order already {o['status']}"}), 400
        items = conn.execute('SELECT * FROM order_items WHERE order_id=?', (oid,)).fetchall()
        for it in items:
            p = conn.execute('SELECT stock, name FROM products WHERE id=?',
                             (it['product_id'],)).fetchone()
            if not p or p['stock'] < it['quantity']:
                return jsonify({'error': f"stock issue for {it['product_name']}"}), 400
        cust = None
        if o['customer_email']:
            cust = conn.execute('SELECT id FROM customers WHERE email=? AND name=?',
                                (o['customer_email'], o['customer_name'])).fetchone()
        if not cust:
            cur = conn.execute(
                'INSERT INTO customers (name, email, phone, address) VALUES (?,?,?,?)',
                (o['customer_name'], o['customer_email'], o['customer_phone'], o['customer_address']))
            cust_id = cur.lastrowid
        else:
            cust_id = cust['id']
        num = _next_invoice_number()
        cur = conn.execute('''INSERT INTO invoices
            (invoice_number, customer_id, invoice_date, due_date,
             subtotal, tax, discount, total, status, notes)
            VALUES (?,?,?,?,?,?,?,?,?,?)''',
            (num, cust_id, datetime.now().strftime('%Y-%m-%d'),
             '', o['subtotal'], o['tax'], o['discount'], o['total'], 'Pending',
             f"From order {o['order_number']}"))
        inv_id = cur.lastrowid
        for it in items:
            conn.execute('''INSERT INTO invoice_items
                (invoice_id, product_id, product_name, quantity, unit_price, total)
                VALUES (?,?,?,?,?,?)''',
                (inv_id, it['product_id'], it['product_name'],
                 it['quantity'], it['unit_price'], it['total']))
            conn.execute('UPDATE products SET stock = stock - ? WHERE id=?',
                         (it['quantity'], it['product_id']))
        conn.execute('UPDATE orders SET status=?, invoice_id=? WHERE id=?',
                     ('Accepted', inv_id, oid))
        conn.commit()
        return jsonify({'ok': True, 'invoice_id': inv_id, 'invoice_number': num})
    except Exception:
        conn.rollback(); raise
    finally:
        conn.close()

@app.route('/api/orders/<int:oid>/reject', methods=['POST'])
def orders_reject(oid):
    conn = get_db()
    try:
        o = conn.execute('SELECT status FROM orders WHERE id=?', (oid,)).fetchone()
        if not o:
            return jsonify({'error': 'not found'}), 404
        if o['status'] != 'Pending':
            return jsonify({'error': f"order already {o['status']}"}), 400
        conn.execute("UPDATE orders SET status='Rejected' WHERE id=?", (oid,))
        conn.commit()
        return jsonify({'ok': True})
    finally:
        conn.close()

print("✅ Catalog + Orders routes ready")

✅ Catalog + Orders routes ready


In [7]:
HTML_SHELL = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Sales & Invoice Tracker</title>
<style>
*{margin:0;padding:0;box-sizing:border-box;font-family:'Segoe UI',Tahoma,sans-serif}
body{background:#f0f2f5;color:#333}
header{background:linear-gradient(135deg,#667eea,#764ba2);color:#fff;padding:18px 30px;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px}
header h1{font-size:22px}
header .subtitle{font-size:12px;opacity:.85}
nav{background:#fff;display:flex;border-bottom:1px solid #ddd;overflow-x:auto}
nav button{background:none;border:none;padding:14px 24px;cursor:pointer;font-size:14px;font-weight:500;color:#666;border-bottom:3px solid transparent;white-space:nowrap}
nav button.active{color:#667eea;border-bottom-color:#667eea;background:#f8f9ff}
nav button:hover{background:#f8f9ff}
nav button .dot{display:inline-block;width:8px;height:8px;background:#ef4444;border-radius:50%;margin-left:6px;vertical-align:middle}
main{padding:22px;max-width:1400px;margin:0 auto}
.view{display:none}.view.active{display:block}
.card{background:#fff;border-radius:10px;padding:20px;margin-bottom:20px;box-shadow:0 2px 8px rgba(0,0,0,.06)}
.stats{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:15px;margin-bottom:20px}
.stat{background:#fff;padding:16px;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,.06);border-left:4px solid #667eea}
.stat .label{font-size:11px;color:#888;text-transform:uppercase}
.stat .value{font-size:22px;font-weight:700;margin-top:4px}
.stat .sub{font-size:11px;color:#999;margin-top:2px}
.stat.green{border-left-color:#10b981}.stat.orange{border-left-color:#f59e0b}
.stat.blue{border-left-color:#3b82f6}.stat.purple{border-left-color:#8b5cf6}
.stat.red{border-left-color:#ef4444}.stat.teal{border-left-color:#14b8a6}
h2{font-size:18px;margin-bottom:14px}h3{font-size:15px;margin-bottom:10px;color:#555}
.btn{padding:8px 16px;border:none;border-radius:6px;cursor:pointer;font-size:13px;font-weight:600}
.btn-primary{background:#667eea;color:#fff}.btn-primary:hover{background:#5568d3}
.btn-success{background:#10b981;color:#fff}.btn-danger{background:#ef4444;color:#fff}
.btn-secondary{background:#e5e7eb;color:#333}.btn-sm{padding:4px 9px;font-size:12px}
.btn:disabled{opacity:.5;cursor:not-allowed}
table{width:100%;border-collapse:collapse;font-size:13px}
th{text-align:left;padding:10px;background:#f8f9fb;color:#555;border-bottom:2px solid #e5e7eb}
td{padding:10px;border-bottom:1px solid #eee}tr:hover{background:#fafafa}
input,select,textarea{width:100%;padding:9px 12px;border:1px solid #d1d5db;border-radius:6px;font-size:13px;font-family:inherit}
input:focus,select:focus,textarea:focus{outline:none;border-color:#667eea}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px}
.fg{margin-bottom:12px}.fg label{display:block;font-size:12px;font-weight:600;color:#555;margin-bottom:4px}
.mo{display:none;position:fixed;inset:0;background:rgba(0,0,0,.5);z-index:1000;align-items:center;justify-content:center;padding:20px}
.mo.active{display:flex}
.modal{background:#fff;border-radius:10px;padding:24px;max-width:700px;width:100%;max-height:90vh;overflow-y:auto}
.modal h3{font-size:18px;margin-bottom:14px}
.ma{display:flex;gap:10px;justify-content:flex-end;margin-top:14px}
.badge{display:inline-block;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:600}
.badge-paid,.badge-accepted{background:#d1fae5;color:#065f46}
.badge-pending{background:#fef3c7;color:#92400e}
.badge-overdue,.badge-rejected{background:#fee2e2;color:#991b1b}
.badge-draft{background:#e5e7eb;color:#374151}
.toolbar{display:flex;justify-content:space-between;align-items:center;margin-bottom:14px;gap:10px;flex-wrap:wrap}
.empty{text-align:center;padding:40px;color:#999}
#toast{position:fixed;bottom:20px;right:20px;padding:12px 20px;border-radius:6px;color:#fff;font-weight:500;font-size:14px;opacity:0;transition:opacity .3s;z-index:2000;pointer-events:none}
#toast.show{opacity:1}#toast.success{background:#10b981}#toast.error{background:#ef4444}
.bar{display:flex;align-items:center;gap:10px;margin-bottom:6px}
.bar-l{width:90px;font-size:12px;color:#666}
.bar-t{flex:1;height:20px;background:#f0f0f0;border-radius:4px;overflow:hidden}
.bar-f{height:100%;background:linear-gradient(90deg,#667eea,#764ba2)}
.bar-v{width:100px;text-align:right;font-size:12px;font-weight:600}
.shop-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(240px,1fr));gap:16px}
.prod-card{background:#fff;border-radius:10px;padding:16px;box-shadow:0 2px 8px rgba(0,0,0,.06);display:flex;flex-direction:column;transition:transform .15s}
.prod-card:hover{transform:translateY(-2px);box-shadow:0 6px 16px rgba(0,0,0,.1)}
.prod-card .cat{font-size:11px;color:#667eea;font-weight:600;text-transform:uppercase;letter-spacing:.5px}
.prod-card h4{font-size:15px;margin:6px 0 4px}
.prod-card .desc{font-size:12px;color:#777;flex:1;margin-bottom:10px}
.prod-card .price{font-size:18px;font-weight:700;color:#333}
.prod-card .stock{font-size:11px;color:#888;margin-top:2px}
.prod-card .row{display:flex;justify-content:space-between;align-items:center;margin-top:8px;gap:8px}
.cart-bar{position:sticky;bottom:0;background:#fff;border-top:2px solid #667eea;padding:14px 22px;box-shadow:0 -4px 12px rgba(0,0,0,.08);display:flex;justify-content:space-between;align-items:center;gap:14px;flex-wrap:wrap}
.cart-info{font-size:14px}
.cart-info strong{font-size:16px;color:#667eea}
#cart-items-mini{font-size:12px;color:#888}
.report-controls{display:flex;gap:10px;flex-wrap:wrap;align-items:flex-end;margin-bottom:16px}
.report-controls .fg{margin-bottom:0;flex:1;min-width:140px}
.report-tabs{display:flex;gap:8px;margin-bottom:16px}
.report-tabs button{padding:8px 18px;border:1px solid #d1d5db;background:#fff;border-radius:6px;cursor:pointer;font-size:13px;font-weight:600;color:#666}
.report-tabs button.active{background:#667eea;color:#fff;border-color:#667eea}
.report-summary{display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr));gap:12px;margin-bottom:18px}
.report-summary .mini{background:#f8f9fb;padding:12px;border-radius:8px}
.report-summary .mini .l{font-size:10px;color:#888;text-transform:uppercase}
.report-summary .mini .v{font-size:18px;font-weight:700;margin-top:2px;color:#333}
.two-col{display:grid;grid-template-columns:1fr 1fr;gap:20px}
@media (max-width:900px){.two-col{grid-template-columns:1fr}}
@media print{body *{visibility:hidden}#invoice-print,#invoice-print *{visibility:visible}
#invoice-print{position:absolute;top:0;left:0;width:100%;padding:30px}}
</style>
</head>
<body>

<header>
  <div>
    <h1>📊 Sales & Invoice Tracker</h1>
    <div class="subtitle">Shop · Orders · Invoices · Reports · Customers · Products</div>
  </div>
  <div id="clock" style="font-size:12px;opacity:.9"></div>
</header>

<nav>
  <button data-view="dashboard" class="active">Dashboard</button>
  <button data-view="shop">🛒 Shop</button>
  <button data-view="orders">📥 Orders <span class="dot" id="orders-dot" style="display:none"></span></button>
  <button data-view="invoices">Invoices</button>
  <button data-view="reports">📈 Reports</button>
  <button data-view="customers">Customers</button>
  <button data-view="products">Products</button>
</nav>

<main>
  <div id="view-dashboard" class="view active">
    <div class="stats">
      <div class="stat green"><div class="label">Total Revenue</div><div class="value" id="s-rev">$0</div><div class="sub">paid invoices</div></div>
      <div class="stat blue"><div class="label">Today</div><div class="value" id="s-today">$0</div><div class="sub" id="s-today-sub">0 invoices</div></div>
      <div class="stat teal"><div class="label">This Month</div><div class="value" id="s-month">$0</div><div class="sub" id="s-month-sub">0 invoices</div></div>
      <div class="stat purple"><div class="label">This Year</div><div class="value" id="s-year">$0</div><div class="sub" id="s-year-sub">0 invoices</div></div>
      <div class="stat orange"><div class="label">Pending Inv.</div><div class="value" id="s-pend">0</div></div>
      <div class="stat red"><div class="label">New Orders</div><div class="value" id="s-orders">0</div></div>
      <div class="stat"><div class="label">Customers</div><div class="value" id="s-cust">0</div></div>
      <div class="stat"><div class="label">Products</div><div class="value" id="s-prod">0</div></div>
    </div>
    <div class="two-col">
      <div class="card"><h3>Monthly Sales (last 6 months)</h3><div id="chart-monthly"></div></div>
      <div class="card"><h3>Top Products</h3><div id="chart-top"></div></div>
    </div>
  </div>

  <div id="view-shop" class="view">
    <div class="card">
      <div class="toolbar">
        <div><h2>🛒 Browse Products</h2><div style="font-size:12px;color:#888">Pick items, then click "Place Order"</div></div>
        <input id="shop-search" placeholder="🔍 Search products…" style="max-width:280px">
      </div>
      <div id="shop-grid" class="shop-grid"></div>
    </div>
  </div>

  <div id="view-orders" class="view">
    <div class="card">
      <div class="toolbar">
        <h2>📥 Incoming Orders</h2>
        <select id="orders-filter" style="max-width:180px">
          <option value="">All</option>
          <option value="Pending" selected>Pending</option>
          <option value="Accepted">Accepted</option>
          <option value="Rejected">Rejected</option>
        </select>
      </div>
      <table><thead><tr><th>#</th><th>Customer</th><th>Items</th><th>Total</th><th>Date</th><th>Status</th><th>Actions</th></tr></thead>
      <tbody id="tb-orders"></tbody></table>
    </div>
  </div>

  <div id="view-invoices" class="view">
    <div class="card">
      <div class="toolbar"><h2>Invoices</h2><button class="btn btn-primary" id="btn-new-invoice">+ New Invoice</button></div>
      <table><thead><tr><th>#</th><th>Customer</th><th>Date</th><th>Due</th><th>Total</th><th>Status</th><th>Actions</th></tr></thead>
      <tbody id="tb-invoices"></tbody></table>
    </div>
  </div>

  <div id="view-reports" class="view">
    <div class="card">
      <h2>📈 Sales Reports</h2>
      <div class="report-tabs">
        <button data-gran="day" class="active">Day-wise</button>
        <button data-gran="month">Month-wise</button>
        <button data-gran="year">Year-wise</button>
      </div>
      <div class="report-controls">
        <div class="fg"><label>Start date</label><input id="rep-start" type="date"></div>
        <div class="fg"><label>End date</label><input id="rep-end" type="date"></div>
        <div class="fg"><label>Limit</label>
          <select id="rep-limit"><option>10</option><option selected>30</option><option>60</option><option>90</option><option>365</option></select>
        </div>
        <button class="btn btn-primary" id="btn-apply-report">Apply</button>
        <button class="btn btn-secondary" id="btn-reset-report">Reset</button>
      </div>
      <div id="report-summary" class="report-summary"></div>
      <div class="two-col">
        <div><h3>Revenue chart</h3><div id="report-chart"></div></div>
        <div><h3>Breakdown</h3>
          <div style="max-height:450px;overflow-y:auto;border:1px solid #eee;border-radius:6px">
            <table><thead><tr><th>Period</th><th>Revenue</th><th>Inv</th><th>Paid</th><th>Orders</th><th>Acc</th></tr></thead>
            <tbody id="tb-report"></tbody></table>
          </div>
        </div>
      </div>
    </div>
  </div>

  <div id="view-customers" class="view">
    <div class="card">
      <div class="toolbar"><h2>Customers</h2><button class="btn btn-primary" id="btn-new-customer">+ New Customer</button></div>
      <table><thead><tr><th>Name</th><th>Email</th><th>Phone</th><th>Address</th><th>Actions</th></tr></thead>
      <tbody id="tb-customers"></tbody></table>
    </div>
  </div>

  <div id="view-products" class="view">
    <div class="card">
      <div class="toolbar"><h2>Products</h2><button class="btn btn-primary" id="btn-new-product">+ New Product</button></div>
      <table><thead><tr><th>Name</th><th>Category</th><th>Price</th><th>Stock</th><th>Actions</th></tr></thead>
      <tbody id="tb-products"></tbody></table>
    </div>
  </div>
</main>

<div id="cart-bar" class="cart-bar" style="display:none">
  <div class="cart-info">
    <strong id="cart-count">0</strong> item(s) · Total: <strong id="cart-total">$0.00</strong>
    <div id="cart-items-mini"></div>
  </div>
  <div style="display:flex;gap:8px">
    <button class="btn btn-secondary" id="btn-clear-cart">Clear</button>
    <button class="btn btn-primary" id="btn-checkout">Place Order →</button>
  </div>
</div>

<div id="mo-checkout" class="mo"><div class="modal">
  <h3>Complete Your Order</h3>
  <div class="fg"><label>Your Name *</label><input id="chk-name"></div>
  <div class="grid">
    <div class="fg"><label>Email</label><input id="chk-email" type="email"></div>
    <div class="fg"><label>Phone</label><input id="chk-phone"></div>
  </div>
  <div class="fg"><label>Shipping Address</label><textarea id="chk-addr" rows="2"></textarea></div>
  <div class="fg"><label>Notes</label><textarea id="chk-notes" rows="2"></textarea></div>
  <div class="ma">
    <button class="btn btn-secondary" data-close="mo-checkout">Cancel</button>
    <button class="btn btn-primary" id="btn-confirm-order">Confirm Order</button>
  </div>
</div></div>

<div id="mo-order" class="mo"><div class="modal" style="max-width:700px">
  <div id="order-view-content"></div>
  <div class="ma">
    <button class="btn btn-secondary" data-close="mo-order">Close</button>
    <button class="btn btn-success" id="btn-accept-order" style="display:none">✓ Accept</button>
    <button class="btn btn-danger" id="btn-reject-order" style="display:none">✕ Reject</button>
  </div>
</div></div>

<div id="mo-customer" class="mo"><div class="modal">
  <h3 id="cust-title">New Customer</h3>
  <input type="hidden" id="cust-id">
  <div class="fg"><label>Name *</label><input id="cust-name"></div>
  <div class="grid">
    <div class="fg"><label>Email</label><input id="cust-email" type="email"></div>
    <div class="fg"><label>Phone</label><input id="cust-phone"></div>
  </div>
  <div class="fg"><label>Address</label><textarea id="cust-addr" rows="2"></textarea></div>
  <div class="ma">
    <button class="btn btn-secondary" data-close="mo-customer">Cancel</button>
    <button class="btn btn-primary" id="btn-save-customer">Save</button>
  </div>
</div></div>

<div id="mo-product" class="mo"><div class="modal">
  <h3 id="prod-title">New Product</h3>
  <input type="hidden" id="prod-id">
  <div class="fg"><label>Name *</label><input id="prod-name"></div>
  <div class="fg"><label>Description</label><textarea id="prod-desc" rows="2"></textarea></div>
  <div class="grid">
    <div class="fg"><label>Price *</label><input id="prod-price" type="number" step="0.01"></div>
    <div class="fg"><label>Stock</label><input id="prod-stock" type="number" value="0"></div>
  </div>
  <div class="fg"><label>Category</label><input id="prod-category" value="General"></div>
  <div class="ma">
    <button class="btn btn-secondary" data-close="mo-product">Cancel</button>
    <button class="btn btn-primary" id="btn-save-product">Save</button>
  </div>
</div></div>

<div id="mo-invoice" class="mo"><div class="modal" style="max-width:900px">
  <h3>New Invoice</h3>
  <div class="grid">
    <div class="fg"><label>Customer *</label><select id="inv-customer"></select></div>
    <div class="fg"><label>Invoice Date</label><input id="inv-date" type="date"></div>
    <div class="fg"><label>Due Date</label><input id="inv-due" type="date"></div>
    <div class="fg"><label>Status</label>
      <select id="inv-status"><option>Pending</option><option>Paid</option><option>Overdue</option><option>Draft</option></select>
    </div>
  </div>
  <h3 style="margin-top:12px">Items</h3>
  <table><thead><tr><th style="width:45%">Product</th><th>Qty</th><th>Price</th><th>Total</th><th></th></tr></thead>
  <tbody id="tb-items"></tbody></table>
  <button class="btn btn-secondary btn-sm" style="margin-top:8px" id="btn-add-item">+ Add Item</button>
  <div class="grid" style="margin-top:12px">
    <div class="fg"><label>Discount ($)</label><input id="inv-disc" type="number" step="0.01" value="0"></div>
    <div class="fg"><label>Tax Rate (%)</label><input id="inv-tax" type="number" step="0.01" value="0"></div>
  </div>
  <div style="background:#f8f9fb;padding:14px;border-radius:8px;margin-top:10px">
    <div style="display:flex;justify-content:space-between;padding:4px 0"><span>Subtotal:</span><span id="sum-sub">$0.00</span></div>
    <div style="display:flex;justify-content:space-between;padding:4px 0"><span>Discount:</span><span id="sum-disc">-$0.00</span></div>
    <div style="display:flex;justify-content:space-between;padding:4px 0"><span>Tax:</span><span id="sum-tax">$0.00</span></div>
    <div style="display:flex;justify-content:space-between;padding:8px 0;border-top:2px solid #ddd;margin-top:6px;font-weight:700;color:#667eea;font-size:16px">
      <span>Total:</span><span id="sum-total">$0.00</span></div>
  </div>
  <div class="fg" style="margin-top:12px"><label>Notes</label><textarea id="inv-notes" rows="2"></textarea></div>
  <div class="ma">
    <button class="btn btn-secondary" data-close="mo-invoice">Cancel</button>
    <button class="btn btn-primary" id="btn-save-invoice">Create Invoice</button>
  </div>
</div></div>

<div id="mo-view" class="mo"><div class="modal" style="max-width:750px">
  <div id="invoice-print"></div>
  <div class="ma">
    <button class="btn btn-secondary" data-close="mo-view">Close</button>
    <button class="btn btn-primary" onclick="window.print()">🖨 Print</button>
  </div>
</div></div>

<div id="toast"></div>

<script>
__JS_PLACEHOLDER__
</script>
</body>
</html>
"""
print("✅ HTML shell ready — length:", len(HTML_SHELL))

✅ HTML shell ready — length: 17909


In [8]:
JS = r"""
'use strict';
const $ = id => document.getElementById(id);
const esc = s => (s == null ? '' : String(s)).replace(/[&<>"']/g,
    c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const money = n => '$' + (Number(n)||0).toFixed(2);

function toast(msg, type='success'){
  const t = $('toast'); t.textContent = msg; t.className = type + ' show';
  clearTimeout(t._t); t._t = setTimeout(() => t.className = type, 2500);
}
async function api(path, opts){
  opts = opts || {};
  const r = await fetch(path, opts);
  let data = {}; try { data = await r.json(); } catch(e){}
  if (!r.ok) throw new Error(data.error || ('HTTP ' + r.status));
  return data;
}

let CUSTOMERS = [], PRODUCTS = [], CATALOG = [], CART = [], CURRENT_ORDER_ID = null;
let REPORT_GRAN = 'day', REPORT_START = '', REPORT_END = '', REPORT_LIMIT = 30;

document.querySelectorAll('nav button').forEach(b => {
  b.addEventListener('click', () => {
    const v = b.dataset.view;
    document.querySelectorAll('nav button').forEach(x => x.classList.remove('active'));
    b.classList.add('active');
    document.querySelectorAll('.view').forEach(x => x.classList.remove('active'));
    $('view-' + v).classList.add('active');
    $('cart-bar').style.display = (v === 'shop') ? 'flex' : 'none';
    if (v === 'dashboard') loadDashboard();
    if (v === 'shop')      loadShop();
    if (v === 'orders')    loadOrders();
    if (v === 'invoices')  loadInvoices();
    if (v === 'reports')   loadReports();
    if (v === 'customers') loadCustomers();
    if (v === 'products')  loadProducts();
  });
});

function openMo(id){ $(id).classList.add('active'); }
function closeMo(id){ $(id).classList.remove('active'); }
document.querySelectorAll('[data-close]').forEach(b =>
  b.addEventListener('click', () => closeMo(b.dataset.close)));

async function loadDashboard(){
  try {
    const d = await api('/api/dashboard');
    $('s-rev').textContent = money(d.revenue);
    $('s-today').textContent = money(d.today_revenue);
    $('s-today-sub').textContent = d.today_invoices + ' invoice(s) today';
    $('s-month').textContent = money(d.month_revenue);
    $('s-month-sub').textContent = d.month_invoices + ' invoice(s) this month';
    $('s-year').textContent = money(d.year_revenue);
    $('s-year-sub').textContent = d.year_invoices + ' invoice(s) this year';
    $('s-pend').textContent = d.pending;
    $('s-orders').textContent = d.orders_pending;
    $('s-cust').textContent = d.customers;
    $('s-prod').textContent = d.products;
    $('orders-dot').style.display = d.orders_pending > 0 ? 'inline-block' : 'none';
    const maxM = Math.max(1, ...d.monthly.map(x=>x.t));
    $('chart-monthly').innerHTML = d.monthly.length ? d.monthly.map(m=>`
      <div class="bar"><div class="bar-l">${esc(m.m)}</div>
      <div class="bar-t"><div class="bar-f" style="width:${(m.t/maxM*100).toFixed(1)}%"></div></div>
      <div class="bar-v">${money(m.t)}</div></div>`).join('') : '<div class="empty">No sales yet</div>';
    const maxT = Math.max(1, ...d.top.map(x=>x.r));
    $('chart-top').innerHTML = d.top.length ? d.top.map(p=>`
      <div class="bar"><div class="bar-l">${esc(p.n)}</div>
      <div class="bar-t"><div class="bar-f" style="width:${(p.r/maxT*100).toFixed(1)}%"></div></div>
      <div class="bar-v">${money(p.r)}</div></div>`).join('') : '<div class="empty">No products sold yet</div>';
  } catch(e){ console.error('dashboard:', e); }
}

document.querySelectorAll('.report-tabs button').forEach(b => {
  b.addEventListener('click', () => {
    document.querySelectorAll('.report-tabs button').forEach(x => x.classList.remove('active'));
    b.classList.add('active');
    REPORT_GRAN = b.dataset.gran;
    loadReports();
  });
});
$('btn-apply-report').addEventListener('click', () => {
  REPORT_START = $('rep-start').value;
  REPORT_END   = $('rep-end').value;
  REPORT_LIMIT = parseInt($('rep-limit').value) || 30;
  if (REPORT_START && REPORT_END && REPORT_START > REPORT_END){
    return toast('Start date must be before end date', 'error');
  }
  loadReports();
});
$('btn-reset-report').addEventListener('click', () => {
  $('rep-start').value = ''; $('rep-end').value = ''; $('rep-limit').value = '30';
  REPORT_START = ''; REPORT_END = ''; REPORT_LIMIT = 30;
  loadReports();
});
async function loadReports(){
  try {
    const params = new URLSearchParams({granularity: REPORT_GRAN, limit: REPORT_LIMIT});
    if (REPORT_START) params.set('start', REPORT_START);
    if (REPORT_END)   params.set('end', REPORT_END);
    const d = await api('/api/reports?' + params.toString());
    const t = d.totals;
    $('report-summary').innerHTML = `
      <div class="mini"><div class="l">Total Revenue</div><div class="v">${money(t.revenue)}</div></div>
      <div class="mini"><div class="l">Invoices</div><div class="v">${t.invoices}</div></div>
      <div class="mini"><div class="l">Paid</div><div class="v" style="color:#10b981">${money(t.paid)}</div></div>
      <div class="mini"><div class="l">Pending</div><div class="v" style="color:#f59e0b">${money(t.pending)}</div></div>
      <div class="mini"><div class="l">Orders</div><div class="v">${t.orders_total}</div></div>
      <div class="mini"><div class="l">Accepted</div><div class="v" style="color:#10b981">${t.orders_accepted}</div></div>`;
    const rows = d.rows || [];
    const maxRev = Math.max(1, ...rows.map(r => r.revenue));
    $('report-chart').innerHTML = rows.length ? rows.slice(0, 20).map(r => `
      <div class="bar"><div class="bar-l">${esc(r.period)}</div>
      <div class="bar-t"><div class="bar-f" style="width:${(r.revenue/maxRev*100).toFixed(1)}%"></div></div>
      <div class="bar-v">${money(r.revenue)}</div></div>`).join('') : '<div class="empty">No data</div>';
    const tb = $('tb-report');
    tb.innerHTML = rows.length ? rows.map(r => `
      <tr><td><strong>${esc(r.period)}</strong></td><td>${money(r.revenue)}</td>
      <td>${r.invoices}</td><td>${money(r.paid)}</td>
      <td>${r.orders_total}</td><td>${r.orders_accepted}</td></tr>`).join('')
      : '<tr><td colspan="6" class="empty">No data</td></tr>';
  } catch(e){ toast('Reports failed: ' + e.message, 'error'); }
}

async function loadShop(){
  try { CATALOG = await api('/api/catalog'); renderShop(); }
  catch(e){ toast('Shop load failed: ' + e.message, 'error'); }
}
function renderShop(){
  const q = ($('shop-search').value || '').toLowerCase().trim();
  const list = q ? CATALOG.filter(p =>
    (p.name + ' ' + p.category + ' ' + (p.description||'')).toLowerCase().includes(q)) : CATALOG;
  const grid = $('shop-grid');
  if (!list.length){ grid.innerHTML = '<div class="empty">No products found</div>'; return; }
  grid.innerHTML = list.map(p => `
    <div class="prod-card">
      <div class="cat">${esc(p.category)}</div>
      <h4>${esc(p.name)}</h4>
      <div class="desc">${esc(p.description)}</div>
      <div class="price">${money(p.price)}</div>
      <div class="stock">${p.stock} in stock</div>
      <div class="row">
        <input type="number" min="1" max="${p.stock}" value="1" id="qty-${p.id}" style="width:70px">
        <button class="btn btn-primary btn-sm" data-add-to-cart="${p.id}">+ Add</button>
      </div>
    </div>`).join('');
}
$('shop-search').addEventListener('input', renderShop);
$('shop-grid').addEventListener('click', (e) => {
  const btn = e.target.closest('[data-add-to-cart]'); if (!btn) return;
  const pid = +btn.dataset.addToCart;
  const p = CATALOG.find(x => x.id === pid); if (!p) return;
  const qty = Math.max(1, parseInt($('qty-' + pid).value) || 1);
  const ex = CART.find(c => c.product_id === pid);
  const newQty = (ex ? ex.quantity : 0) + qty;
  if (newQty > p.stock) return toast('Only ' + p.stock + ' available', 'error');
  if (ex) ex.quantity = newQty;
  else CART.push({product_id: p.id, product_name: p.name, unit_price: p.price, quantity: qty});
  renderCart(); toast('Added: ' + p.name);
});
function renderCart(){
  const count = CART.reduce((s,c)=>s+c.quantity, 0);
  const total = CART.reduce((s,c)=>s+c.quantity*c.unit_price, 0);
  $('cart-count').textContent = count;
  $('cart-total').textContent = money(total);
  $('cart-items-mini').textContent = CART.map(c => `${c.quantity}× ${c.product_name}`).join(' · ');
  $('cart-bar').style.display = CART.length ? 'flex' : 'none';
}
$('btn-clear-cart').addEventListener('click', () => { CART = []; renderCart(); toast('Cart cleared'); });
$('btn-checkout').addEventListener('click', () => {
  if (!CART.length) return toast('Cart is empty', 'error');
  ['chk-name','chk-email','chk-phone','chk-addr','chk-notes'].forEach(id => $(id).value = '');
  openMo('mo-checkout');
});
$('btn-confirm-order').addEventListener('click', async () => {
  const name = $('chk-name').value.trim();
  if (!name) return toast('Name is required', 'error');
  const payload = {
    customer_name: name,
    customer_email: $('chk-email').value.trim(),
    customer_phone: $('chk-phone').value.trim(),
    customer_address: $('chk-addr').value.trim(),
    notes: $('chk-notes').value.trim(),
    items: CART.map(c => ({product_id: c.product_id, product_name: c.product_name,
                            quantity: c.quantity, unit_price: c.unit_price}))
  };
  try {
    const res = await api('/api/orders', {method:'POST', headers:{'Content-Type':'application/json'}, body: JSON.stringify(payload)});
    toast('Order ' + res.order_number + ' placed');
    CART = []; renderCart(); closeMo('mo-checkout');
    await loadShop(); loadDashboard();
  } catch(e){ toast('Error: ' + e.message, 'error'); }
});

async function loadOrders(){
  try {
    const filter = $('orders-filter').value;
    const url = filter ? '/api/orders?status=' + filter : '/api/orders';
    const list = await api(url);
    const tb = $('tb-orders');
    if (!list.length){ tb.innerHTML = '<tr><td colspan="7" class="empty">No orders</td></tr>'; return; }
    tb.innerHTML = list.map(o => `
      <tr><td><strong>${esc(o.order_number)}</strong></td><td>${esc(o.customer_name)}</td>
      <td id="ord-items-${o.id}">…</td><td><strong>${money(o.total)}</strong></td>
      <td>${esc(o.created_at || '')}</td>
      <td><span class="badge badge-${(o.status||'pending').toLowerCase()}">${esc(o.status)}</span></td>
      <td><button class="btn btn-secondary btn-sm" data-view-order="${o.id}">View</button>
      ${o.status === 'Pending' ? `
        <button class="btn btn-success btn-sm" data-accept-order="${o.id}">✓</button>
        <button class="btn btn-danger btn-sm" data-reject-order="${o.id}">✕</button>` : ''}</td></tr>`).join('');
    for (const o of list){
      try {
        const d = await api('/api/orders/' + o.id);
        const el = $('ord-items-' + o.id);
        if (el) el.textContent = d.items.map(i => `${i.quantity}× ${i.product_name}`).join(', ');
      } catch(e){}
    }
  } catch(e){ toast('Orders load failed: ' + e.message, 'error'); }
}
$('orders-filter').addEventListener('change', loadOrders);
$('tb-orders').addEventListener('click', async (e) => {
  const t = e.target;
  if (t.dataset.viewOrder) viewOrder(+t.dataset.viewOrder);
  else if (t.dataset.acceptOrder) await acceptOrder(+t.dataset.acceptOrder);
  else if (t.dataset.rejectOrder){ if (!confirm('Reject this order?')) return; await rejectOrder(+t.dataset.rejectOrder); }
});
async function acceptOrder(id){
  try {
    const res = await api('/api/orders/' + id + '/accept', {method:'POST'});
    toast('Accepted → invoice ' + res.invoice_number);
    closeMo('mo-order'); loadOrders(); loadDashboard(); loadInvoices();
  } catch(e){ toast('Error: ' + e.message, 'error'); }
}
async function rejectOrder(id){
  try {
    await api('/api/orders/' + id + '/reject', {method:'POST'});
    toast('Order rejected'); closeMo('mo-order'); loadOrders(); loadDashboard();
  } catch(e){ toast('Error: ' + e.message, 'error'); }
}
async function viewOrder(id){
  try {
    const o = await api('/api/orders/' + id);
    CURRENT_ORDER_ID = id;
    const rows = o.items.map(it => `
      <tr><td>${esc(it.product_name)}</td><td>${it.quantity}</td>
      <td>${money(it.unit_price)}</td><td>${money(it.total)}</td></tr>`).join('');
    $('order-view-content').innerHTML = `
      <div style="display:flex;justify-content:space-between;margin-bottom:14px">
        <div><h2 style="color:#667eea;margin:0">ORDER</h2>
          <div style="color:#666;font-size:13px">${esc(o.order_number)}</div></div>
        <span class="badge badge-${(o.status||'pending').toLowerCase()}" style="font-size:13px;padding:6px 14px">${esc(o.status)}</span>
      </div>
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:16px;font-size:13px">
        <div><div style="color:#888;font-size:11px">CUSTOMER</div>
          <div style="font-weight:600">${esc(o.customer_name)}</div>
          <div>${esc(o.customer_email)}</div><div>${esc(o.customer_phone)}</div>
          <div>${esc(o.customer_address)}</div></div>
        <div style="text-align:right"><div><b>Placed:</b> ${esc(o.created_at||'')}</div></div>
      </div>
      <table style="margin-bottom:14px">
        <thead><tr><th>Item</th><th>Qty</th><th>Price</th><th>Total</th></tr></thead>
        <tbody>${rows}</tbody></table>
      <div style="background:#f8f9fb;padding:14px;border-radius:8px">
        <div style="display:flex;justify-content:space-between;padding:4px 0"><span>Subtotal:</span><span>${money(o.subtotal)}</span></div>
        <div style="display:flex;justify-content:space-between;padding:4px 0"><span>Tax:</span><span>${money(o.tax)}</span></div>
        <div style="display:flex;justify-content:space-between;padding:8px 0;border-top:2px solid #ddd;margin-top:6px;font-weight:700;color:#667eea;font-size:16px">
          <span>Total:</span><span>${money(o.total)}</span></div>
      </div>
      ${o.notes ? `<div style="margin-top:12px;padding:10px;background:#f8f9fb;border-radius:6px;font-size:13px"><b>Notes:</b> ${esc(o.notes)}</div>` : ''}`;
    $('btn-accept-order').style.display = o.status === 'Pending' ? 'inline-block' : 'none';
    $('btn-reject-order').style.display = o.status === 'Pending' ? 'inline-block' : 'none';
    openMo('mo-order');
  } catch(e){ toast('Error: ' + e.message, 'error'); }
}
$('btn-accept-order').addEventListener('click', () => CURRENT_ORDER_ID && acceptOrder(CURRENT_ORDER_ID));
$('btn-reject-order').addEventListener('click', () => {
  if (!CURRENT_ORDER_ID) return;
  if (!confirm('Reject this order?')) return;
  rejectOrder(CURRENT_ORDER_ID);
});

async function loadCustomers(){
  try {
    CUSTOMERS = await api('/api/customers');
    const tb = $('tb-customers');
    if (!CUSTOMERS.length){ tb.innerHTML = '<tr><td colspan="5" class="empty">No customers yet</td></tr>'; return; }
    tb.innerHTML = CUSTOMERS.map(c => `
      <tr><td><strong>${esc(c.name)}</strong></td><td>${esc(c.email)}</td>
      <td>${esc(c.phone)}</td><td>${esc(c.address)}</td>
      <td><button class="btn btn-secondary btn-sm" data-edit-customer="${c.id}">Edit</button>
      <button class="btn btn-danger btn-sm" data-del-customer="${c.id}">Delete</button></td></tr>`).join('');
  } catch(e){ toast('Load failed: ' + e.message, 'error'); }
}
$('btn-new-customer').addEventListener('click', () => {
  $('cust-title').textContent = 'New Customer'; $('cust-id').value = '';
  ['cust-name','cust-email','cust-phone','cust-addr'].forEach(id => $(id).value = '');
  openMo('mo-customer');
});
$('btn-save-customer').addEventListener('click', async () => {
  const id = $('cust-id').value;
  const payload = {
    name: $('cust-name').value.trim(), email: $('cust-email').value.trim(),
    phone: $('cust-phone').value.trim(), address: $('cust-addr').value.trim()
  };
  if (!payload.name) return toast('Name is required', 'error');
  try {
    if (id) await api('/api/customers/' + id, {method:'PUT', headers:{'Content-Type':'application/json'}, body: JSON.stringify(payload)});
    else    await api('/api/customers', {method:'POST', headers:{'Content-Type':'application/json'}, body: JSON.stringify(payload)});
    toast('Customer saved'); closeMo('mo-customer'); await loadCustomers(); loadDashboard();
  } catch(e){ toast('Error: ' + e.message, 'error'); }
});
$('tb-customers').addEventListener('click', async (e) => {
  const t = e.target;
  if (t.dataset.editCustomer){
    const c = CUSTOMERS.find(x => x.id === +t.dataset.editCustomer);
    if (!c) return;
    $('cust-title').textContent = 'Edit Customer';
    $('cust-id').value = c.id;
    $('cust-name').value = c.name || ''; $('cust-email').value = c.email || '';
    $('cust-phone').value = c.phone || ''; $('cust-addr').value = c.address || '';
    openMo('mo-customer');
  } else if (t.dataset.delCustomer){
    if (!confirm('Delete this customer?')) return;
    try { await api('/api/customers/' + t.dataset.delCustomer, {method:'DELETE'});
      toast('Customer deleted'); await loadCustomers(); loadDashboard();
    } catch(err){ toast('Error: ' + err.message, 'error'); }
  }
});

async function loadProducts(){
  try {
    PRODUCTS = await api('/api/products');
    const tb = $('tb-products');
    if (!PRODUCTS.length){ tb.innerHTML = '<tr><td colspan="5" class="empty">No products</td></tr>'; return; }
    tb.innerHTML = PRODUCTS.map(p => `
      <tr><td><strong>${esc(p.name)}</strong></td><td>${esc(p.category)}</td>
      <td>${money(p.price)}</td><td>${p.stock}</td>
      <td><button class="btn btn-secondary btn-sm" data-edit-product="${p.id}">Edit</button>
      <button class="btn btn-danger btn-sm" data-del-product="${p.id}">Delete</button></td></tr>`).join('');
  } catch(e){ toast('Load failed: ' + e.message, 'error'); }
}
$('btn-new-product').addEventListener('click', () => {
  $('prod-title').textContent = 'New Product'; $('prod-id').value = '';
  ['prod-name','prod-desc','prod-price','prod-category'].forEach(id => $(id).value = '');
  $('prod-category').value = 'General'; $('prod-stock').value = '0';
  openMo('mo-product');
});
$('btn-save-product').addEventListener('click', async () => {
  const id = $('prod-id').value;
  const payload = {
    name: $('prod-name').value.trim(), description: $('prod-desc').value.trim(),
    price: $('prod-price').value || 0, stock: $('prod-stock').value || 0,
    category: $('prod-category').value.trim() || 'General'
  };
  if (!payload.name) return toast('Name is required', 'error');
  try {
    if (id) await api('/api/products/' + id, {method:'PUT', headers:{'Content-Type':'application/json'}, body: JSON.stringify(payload)});
    else    await api('/api/products', {method:'POST', headers:{'Content-Type':'application/json'}, body: JSON.stringify(payload)});
    toast('Product saved'); closeMo('mo-product'); await loadProducts(); loadDashboard();
  } catch(e){ toast('Error: ' + e.message, 'error'); }
});
$('tb-products').addEventListener('click', async (e) => {
  const t = e.target;
  if (t.dataset.editProduct){
    const p = PRODUCTS.find(x => x.id === +t.dataset.editProduct);
    if (!p) return;
    $('prod-title').textContent = 'Edit Product';
    $('prod-id').value = p.id;
    $('prod-name').value = p.name || ''; $('prod-desc').value = p.description || '';
    $('prod-price').value = p.price; $('prod-stock').value = p.stock;
    $('prod-category').value = p.category || 'General';
    openMo('mo-product');
  } else if (t.dataset.delProduct){
    if (!confirm('Delete this product?')) return;
    try { await api('/api/products/' + t.dataset.delProduct, {method:'DELETE'});
      toast('Product deleted'); await loadProducts(); loadDashboard();
    } catch(err){ toast('Error: ' + err.message, 'error'); }
  }
});

async function loadInvoices(){
  try {
    const list = await api('/api/invoices');
    const tb = $('tb-invoices');
    if (!list.length){ tb.innerHTML = '<tr><td colspan="7" class="empty">No invoices yet</td></tr>'; return; }
    tb.innerHTML = list.map(i => `
      <tr><td><strong>${esc(i.invoice_number)}</strong></td>
      <td>${esc(i.customer_name || '—')}</td>
      <td>${esc(i.invoice_date)}</td><td>${esc(i.due_date)}</td>
      <td><strong>${money(i.total)}</strong></td>
      <td><span class="badge badge-${(i.status||'draft').toLowerCase()}">${esc(i.status)}</span></td>
      <td><button class="btn btn-secondary btn-sm" data-view-invoice="${i.id}">View</button>
      <button class="btn btn-success btn-sm" data-pay-invoice="${i.id}">Pay</button>
      <button class="btn btn-danger btn-sm" data-del-invoice="${i.id}">Del</button></td></tr>`).join('');
  } catch(e){ toast('Load failed: ' + e.message, 'error'); }
}
$('tb-invoices').addEventListener('click', async (e) => {
  const t = e.target;
  if (t.dataset.viewInvoice) viewInvoice(+t.dataset.viewInvoice);
  else if (t.dataset.payInvoice){
    try { await api('/api/invoices/' + t.dataset.payInvoice + '/status',
      {method:'PUT', headers:{'Content-Type':'application/json'}, body: JSON.stringify({status:'Paid'})});
      toast('Marked Paid'); await loadInvoices(); loadDashboard();
    } catch(err){ toast('Error: ' + err.message, 'error'); }
  } else if (t.dataset.delInvoice){
    if (!confirm('Delete this invoice?')) return;
    try { await api('/api/invoices/' + t.dataset.delInvoice, {method:'DELETE'});
      toast('Invoice deleted'); await loadInvoices(); loadDashboard();
    } catch(err){ toast('Error: ' + err.message, 'error'); }
  }
});
$('btn-new-invoice').addEventListener('click', async () => {
  try {
    if (!CUSTOMERS.length) CUSTOMERS = await api('/api/customers');
    if (!PRODUCTS.length) PRODUCTS = await api('/api/products');
    if (!CUSTOMERS.length) return toast('Add a customer first', 'error');
    if (!PRODUCTS.length) return toast('Add a product first', 'error');
    $('inv-customer').innerHTML = CUSTOMERS.map(c => `<option value="${c.id}">${esc(c.name)}</option>`).join('');
    $('inv-date').value = new Date().toISOString().slice(0,10);
    $('inv-due').value = ''; $('inv-status').value = 'Pending';
    $('inv-disc').value = '0'; $('inv-tax').value = '0'; $('inv-notes').value = '';
    $('tb-items').innerHTML = ''; addInvoiceRow(); calcTotals();
    openMo('mo-invoice');
  } catch(e){ toast('Error: ' + e.message, 'error'); }
});
function addInvoiceRow(){
  const row = document.createElement('tr');
  const opts = PRODUCTS.map(p =>
    `<option value="${p.id}" data-price="${p.price}" data-name="${esc(p.name)}">${esc(p.name)} — ${money(p.price)}</option>`).join('');
  row.innerHTML = `
    <td><select class="it-prod"><option value="">-- select --</option>${opts}</select></td>
    <td><input type="number" class="it-qty" value="1" min="1"></td>
    <td><input type="number" class="it-price" step="0.01" value="0"></td>
    <td class="it-total">$0.00</td>
    <td><button class="btn btn-danger btn-sm" data-remove-row>×</button></td>`;
  row.querySelector('.it-prod').addEventListener('change', ev => {
    const o = ev.target.selectedOptions[0];
    row.querySelector('.it-price').value = o.dataset.price || 0; calcTotals();
  });
  row.querySelector('.it-qty').addEventListener('input', calcTotals);
  row.querySelector('.it-price').addEventListener('input', calcTotals);
  row.querySelector('[data-remove-row]').addEventListener('click', () => { row.remove(); calcTotals(); });
  $('tb-items').appendChild(row);
}
$('btn-add-item').addEventListener('click', addInvoiceRow);
$('inv-disc').addEventListener('input', calcTotals);
$('inv-tax').addEventListener('input', calcTotals);
function calcTotals(){
  let sub = 0;
  document.querySelectorAll('#tb-items tr').forEach(row => {
    const q = parseFloat(row.querySelector('.it-qty').value) || 0;
    const p = parseFloat(row.querySelector('.it-price').value) || 0;
    const t = q * p; row.querySelector('.it-total').textContent = money(t); sub += t;
  });
  const disc = parseFloat($('inv-disc').value) || 0;
  const rate = parseFloat($('inv-tax').value) || 0;
  const taxable = Math.max(0, sub - disc);
  const tax = taxable * rate / 100;
  $('sum-sub').textContent = money(sub);
  $('sum-disc').textContent = '-' + money(disc);
  $('sum-tax').textContent = money(tax);
  $('sum-total').textContent = money(taxable + tax);
}
$('btn-save-invoice').addEventListener('click', async () => {
  const cid = $('inv-customer').value;
  if (!cid) return toast('Select a customer', 'error');
  const items = [];
  document.querySelectorAll('#tb-items tr').forEach(row => {
    const sel = row.querySelector('.it-prod');
    if (!sel.value) return;
    const o = sel.selectedOptions[0];
    items.push({ product_id: +sel.value, product_name: o.dataset.name,
      quantity: parseInt(row.querySelector('.it-qty').value) || 0,
      unit_price: parseFloat(row.querySelector('.it-price').value) || 0 });
  });
  if (!items.length) return toast('Add at least one item', 'error');
  const payload = {
    customer_id: +cid, invoice_date: $('inv-date').value, due_date: $('inv-due').value,
    status: $('inv-status').value, discount: parseFloat($('inv-disc').value) || 0,
    tax_rate: parseFloat($('inv-tax').value) || 0, notes: $('inv-notes').value, items
  };
  try {
    const res = await api('/api/invoices', {method:'POST', headers:{'Content-Type':'application/json'}, body: JSON.stringify(payload)});
    toast('Invoice ' + res.invoice_number + ' created');
    closeMo('mo-invoice'); await loadInvoices(); loadDashboard();
  } catch(e){ toast('Error: ' + e.message, 'error'); }
});
async function viewInvoice(id){
  try {
    const inv = await api('/api/invoices/' + id);
    const badge = 'badge-' + (inv.status || 'draft').toLowerCase();
    const rows = inv.items.map(it => `
      <tr><td>${esc(it.product_name)}</td><td>${it.quantity}</td>
      <td>${money(it.unit_price)}</td><td>${money(it.total)}</td></tr>`).join('');
    $('invoice-print').innerHTML = `
      <div style="display:flex;justify-content:space-between;margin-bottom:18px">
        <div><h2 style="color:#667eea;margin:0">INVOICE</h2>
          <div style="color:#666;font-size:13px">${esc(inv.invoice_number)}</div></div>
        <span class="badge ${badge}" style="font-size:13px;padding:6px 14px">${esc(inv.status)}</span>
      </div>
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:16px;font-size:13px">
        <div><div style="color:#888;font-size:11px">BILL TO</div>
          <div style="font-weight:600">${esc(inv.customer_name)}</div>
          <div>${esc(inv.customer_email)}</div><div>${esc(inv.customer_phone)}</div>
          <div>${esc(inv.customer_address)}</div></div>
        <div style="text-align:right">
          <div><b>Date:</b> ${esc(inv.invoice_date)}</div>
          <div><b>Due:</b> ${esc(inv.due_date)}</div></div>
      </div>
      <table style="margin-bottom:14px">
        <thead><tr><th>Item</th><th>Qty</th><th>Price</th><th>Total</th></tr></thead>
        <tbody>${rows}</tbody></table>
      <div style="background:#f8f9fb;padding:14px;border-radius:8px">
        <div style="display:flex;justify-content:space-between;padding:4px 0"><span>Subtotal:</span><span>${money(inv.subtotal)}</span></div>
        <div style="display:flex;justify-content:space-between;padding:4px 0"><span>Discount:</span><span>-${money(inv.discount)}</span></div>
        <div style="display:flex;justify-content:space-between;padding:4px 0"><span>Tax:</span><span>${money(inv.tax)}</span></div>
        <div style="display:flex;justify-content:space-between;padding:8px 0;border-top:2px solid #ddd;margin-top:6px;font-weight:700;color:#667eea;font-size:16px">
          <span>Total:</span><span>${money(inv.total)}</span></div>
      </div>
      ${inv.notes ? `<div style="margin-top:14px;padding:10px;background:#f8f9fb;border-radius:6px;font-size:13px"><b>Notes:</b> ${esc(inv.notes)}</div>` : ''}`;
    openMo('mo-view');
  } catch(e){ toast('Error: ' + e.message, 'error'); }
}

function tickClock(){ $('clock').textContent = new Date().toLocaleString(); }
setInterval(tickClock, 1000); tickClock();

(async function boot(){
  try {
    await loadCustomers(); await loadProducts();
    await loadInvoices(); await loadDashboard();
    console.log('✅ App booted');
  } catch(e){ console.error('boot:', e); }
})();

setInterval(() => {
  if ($('view-dashboard').classList.contains('active')) loadDashboard();
  if ($('view-orders').classList.contains('active')) loadOrders();
}, 5000);
"""

FULL_HTML = HTML_SHELL.replace('__JS_PLACEHOLDER__', JS)

@app.route('/')
def index():
    return FULL_HTML

print("✅ / route mounted — total HTML:", len(FULL_HTML), "chars")

✅ / route mounted — total HTML: 46171 chars


In [9]:
def run_app():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False, threaded=True)

if 'server_thread' in globals() and server_thread.is_alive():
    print("⚠️  Server already running")
else:
    server_thread = threading.Thread(target=run_app, daemon=True)
    server_thread.start()
    time.sleep(2)
    print("✅ Flask server started on 0.0.0.0:5000")

print("\n" + "="*60)
print("📱 HOW TO ACCESS THE APP")
print("="*60)
print(f"  💻 This machine        :  http://localhost:5000")
print(f"  📡 Other devices (LAN) :  http://{LOCAL_IP}:5000")
print("="*60)
print("  ⚠️  If LAN URL doesn't open, open port 5000:")
print("    Linux :  sudo ufw allow 5000/tcp")
print("    Windows (Admin PowerShell):")
print("             New-NetFirewallRule -DisplayName 'Flask 5000' -Direction Inbound -LocalPort 5000 -Protocol TCP -Action Allow")
print("    macOS :  System Settings → Network → Firewall → Options → allow python")
print("="*60 + "\n")

try:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(5000)")
    print(f"🌐 Colab proxy URL: {url}")
    from IPython.display import display, HTML as IPHTML
    display(IPHTML(f'<a href="{url}" target="_blank">👉 Open in Colab</a>'))
except ImportError:
    webbrowser.open('http://localhost:5000')
    print("🌐 Opening http://localhost:5000 in this machine's browser")

 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on all addresses.
 * Running on http://192.168.29.57:5000/ (Press CTRL+C to quit)


✅ Flask server started on 0.0.0.0:5000

📱 HOW TO ACCESS THE APP
  💻 This machine        :  http://localhost:5000
  📡 Other devices (LAN) :  http://192.168.29.57:5000
  ⚠️  If LAN URL doesn't open, open port 5000:
    Linux :  sudo ufw allow 5000/tcp
    Windows (Admin PowerShell):
             New-NetFirewallRule -DisplayName 'Flask 5000' -Direction Inbound -LocalPort 5000 -Protocol TCP -Action Allow
    macOS :  System Settings → Network → Firewall → Options → allow python

🌐 Opening http://localhost:5000 in this machine's browser


127.0.0.1 - - [12/Nov/2026 15:18:36] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 15:18:36] "GET /api/customers HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 15:18:36] "GET /api/products HTTP/1.1" 200 -
127.0.0.1 - - [12/Nov/2026 15:18:36] "GET /api/invoices HTTP/1.1" 200 -
Traceback (most recent call last):
  File "C:\Users\AZIZUR RAHAMAN\anaconda3\lib\site-packages\flask\app.py", line 1950, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\AZIZUR RAHAMAN\anaconda3\lib\site-packages\flask\app.py", line 1926, in dispatch_request
    self.raise_routing_exception(req)
  File "C:\Users\AZIZUR RAHAMAN\anaconda3\lib\site-packages\flask\app.py", line 1908, in raise_routing_exception
    raise request.routing_exception
  File "C:\Users\AZIZUR RAHAMAN\anaconda3\lib\site-packages\flask\ctx.py", line 350, in match_request
    result = self.url_adapter.match(return_rule=True)
  File "C:\Users\AZIZUR RAHAMAN\anaconda3\lib\site-packages\werkzeug\routing.py", line 2042, in